In [ ]:
import numpy as np
import scipy.stats
import json
from collections import defaultdict
from scipy import spatial, stats
import matplotlib.pyplot as plt
from collections import Counter
from validate_semantics import load_wordnet_axes, get_glove_vecs
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from scipy.stats import mannwhitneyu
from sklearn.metrics.pairwise import cosine_similarity
import random
import pandas as pd
import math

In [ ]:
ROOT = '/mnt/data0/lucy/manosphere/'
DATA = ROOT + 'data/'
GLOVE = DATA + 'glove/'
LOGS = ROOT + 'logs/'

### Axes in vector space

In [ ]:
axes, axes_vocab = load_wordnet_axes()
glove_vecs = get_glove_vecs(set(), axes_vocab, exp_name='default')

In [ ]:
def show_top_axes(path, scores, show_bias=True, show_effect=True):
    with open(path, 'r') as infile:
        biases = json.load(infile) # {c : { pole : (bias, effect) } }
        
    for c in biases:
        if not show_bias: continue
        print("**** BIAS ****", c)
        vals = []
        dims = []
        for pole in biases[c]: 
            p_val, effect, b_t_f1, b_t_f2 = biases[c][pole]
            # separability
            if scores[pole + '_left'][0] >= 0 and scores[pole + '_right'][0] >= 0:
                dims.append((pole, round(b_t_f2, 3)))
                vals.append(abs(b_t_f2))
        sorted_dims = [x for _,x in sorted(zip(vals,dims))]
        for i in range(1, 6): 
            pole, bias = sorted_dims[-i]
            left, right = axes[pole]
            p_val, effect, b_t_f1, b_t_f2 = biases[c][pole]
            if b_t_f2 > 0: 
                this_side = left
            elif b_t_f2 <= 0: 
                this_side = right
            random.seed(0)
            adj_sample = random.sample(this_side, min(len(this_side), 10))
            print(bias, pole, ', '.join(adj_sample))
        print()

    for c in biases: 
        if not show_effect: continue
        print("**** DIFF ****", c)
        vals = []
        dims = []
        for pole in biases[c]: 
            p_val, effect, b_t_f1, b_t_f2 = biases[c][pole]
            # separability + statistical significance
            if scores[pole + '_left'][0] >= 0 and scores[pole + '_right'][0] >= 0 and p_val < 0.001:
                dims.append((pole, round(b_t_f2, 3)))
                vals.append(abs(effect))
        sorted_dims = [x for _,x in sorted(zip(vals,dims))]
        num_examples = min(len(sorted_dims)+1, 6)
        for i in range(1, num_examples): 
            pole, bias = sorted_dims[-i]
            left, right = axes[pole]
            p_val, effect, b_t_f1, b_t_f2 = biases[c][pole]
            if b_t_f2 > 0 and effect > 0: 
                this_side = left
                degree = 'more'
            elif b_t_f2 > 0 and effect <= 0: 
                this_side = left
                degree = 'less'
            elif b_t_f2 <= 0 and effect > 0: 
                this_side = right
                degree = 'less'
            elif b_t_f2 <= 0 and effect <= 0: 
                this_side = right
                degree = 'more'
            random.seed(0)
            adj_sample = random.sample(this_side, min(len(this_side), 10))
            print(round(effect, 3), bias, pole, degree.upper(), ', '.join(adj_sample))
        print()

In [ ]:
def get_occ_sent_counts(): 
    with open(DATA + 'semantics/occupation_sents.json', 'r') as infile: 
        occ_sents = json.load(infile)
    with open(DATA + 'semantics/cleaned/occupations.json', 'r') as infile: 
        lexicon_dict = json.load(infile)
    occ_cat = defaultdict(list)
    for c in lexicon_dict: 
        for occ in lexicon_dict[c]['high']: 
            occ_cat[occ].append(c)
    occ_cat_counts = defaultdict(Counter)
    for occ in occ_sents: 
        for cat in occ_cat[occ]: 
            occ_cat_counts[cat][occ] = len(occ_sents[occ])
    avg = []
    num_sents = []
    for cat in occ_cat_counts: 
        print(cat.upper(), len(occ_cat_counts[cat]))
        avg.append(len(occ_cat_counts[cat]))
        for occ in occ_cat_counts[cat]:
            num_sents.append(occ_cat_counts[cat][occ])
    print("\nMEAN:", np.mean(avg))
    print('\nTOTAL:', np.sum(avg))
    print("\nTOTAL SENTS:", np.sum(num_sents))

In [ ]:
get_occ_sent_counts()

### Occupation and self-consistency evaluation

In [ ]:
def mean_confidence_interval(data, confidence=0.90):
    '''
    copied from https://stackoverflow.com/questions/15033511/compute-a-confidence-interval-from-sample-data
    '''
    a = 1.0 * np.array(data)
    n = len(a)
    m, se = np.mean(a), scipy.stats.sem(a)
    h = se * scipy.stats.t.ppf((1 + confidence) / 2., n-1)
    return m, m-h, m+h

def get_ci(a): 
    a = 1.0 * np.array(a)
    n = len(a)
    m, se = np.mean(a), scipy.stats.sem(a)
    h = se * scipy.stats.t.ppf((1 + 0.95) / 2., n-1)
    return h

def plot_axes_quality(exp_name, do_plot=True): 
    quality_file_path = LOGS + 'semantics_val/axes_quality_' + exp_name + '.txt'
    scores = defaultdict(dict) # {synset: {word : (predicted, true)}}
    with open(quality_file_path, 'r') as infile: 
        for line in infile: 
            contents = line.strip().split('\t')
            scores[contents[0]][contents[1]] = (float(contents[2]), contents[3])
    avg_scores = Counter()
    ci_scores = Counter()
    good_count = 0
    for synset in scores: 
        left_scores = []
        right_scores = []
        for w in scores[synset]: 
            if scores[synset][w][1] == 'left': 
                left_scores.append(-1*scores[synset][w][0])
            else: 
                right_scores.append(scores[synset][w][0])
        if left_scores != []: 
            # some are empty since they only had one word with reps
            avg_scores[synset + '_left'] = np.mean(left_scores) 
            ci_scores[synset + '_left'] = mean_confidence_interval(left_scores)
        if right_scores != []: 
            avg_scores[synset + '_right'] = np.mean(right_scores) 
            ci_scores[synset + '_right'] = mean_confidence_interval(right_scores)
        if avg_scores[synset + '_left'] >= 0 and avg_scores[synset + '_right'] >= 0: 
            good_count += 1
    ordered_scores = avg_scores.most_common()
    avg_sim = np.mean(list(avg_scores.values()))
    if do_plot: 
        print("BEST", ordered_scores[:10])
        print("WORST", ordered_scores[::-1][:10])
        print("AVG SIMILARITY TO CORRECT SIDE:", avg_sim)
        print("95\% CI:", get_ci(list(avg_scores.values())))
        print("GOOD AXES COUNT:", good_count)
        plt.hist(avg_scores.values(), density=True, bins=20)
        plt.axvline(x=np.mean(list(avg_scores.values())), color='red')
        plt.ylabel('frequency')
        plt.xlabel('similarity to correct side')
        plt.title(exp_name)
        plt.show()
    return ci_scores, avg_sim, good_count

In [ ]:
# GloVe - zscore
scores1, _, _ = plot_axes_quality('glove-zscore')

In [ ]:
# GloVe 
scores1, _, _ = plot_axes_quality('default')

In [ ]:
show_top_axes(LOGS + 'semantics_val/occupations/frameaxis_default.json', scores1, show_effect=False)

In [ ]:
scores2, _, _ = plot_axes_quality('bert-default')

In [ ]:
show_top_axes(LOGS + 'semantics_val/occupations/frameaxis_bert-default.json', scores2, show_effect=False)

In [ ]:
show_top_axes(LOGS + 'semantics_val/person/frameaxis_bert-default.json', scores2, show_bias=False)

In [ ]:
scores3, _, _ = plot_axes_quality('bert-zscore')

In [ ]:
show_top_axes(LOGS + 'semantics_val/occupations/frameaxis_bert-zscore.json', scores3, show_effect=False)

In [ ]:
show_top_axes(LOGS + 'semantics_val/person/frameaxis_bert-zscore.json', scores3, show_bias=False)

In [ ]:
scores7, _, _ = plot_axes_quality('bert-base-prob')

In [ ]:
show_top_axes(LOGS + 'semantics_val/occupations/frameaxis_bert-base-prob.json', scores7, show_effect=False)

In [ ]:
show_top_axes(LOGS + 'semantics_val/person/frameaxis_bert-base-prob.json', scores7, show_bias=False)

In [ ]:
scores8, _, _ = plot_axes_quality('bert-base-prob-zscore')

In [ ]:
show_top_axes(LOGS + 'semantics_val/occupations/frameaxis_bert-base-prob-zscore.json', scores8, show_effect=False)

In [ ]:
show_top_axes(LOGS + 'semantics_val/person/frameaxis_bert-base-prob-zscore.json', scores8, show_bias=False)

In [ ]:
# U-tests
# z-scored bert prob vs. glove
print(mannwhitneyu(list(scores8.values()), list(scores1.values())))
# z-scored bert prob vs. z-scored bert random
print(mannwhitneyu(list(scores8.values()), list(scores3.values())))
# z-scored bert random vs. glove
print(mannwhitneyu(list(scores3.values()), list(scores1.values())))

# Human evaluation task materials

This section includes metadata used to design the evaluation task, including: 
- example occupations in each category 
- example sentences for each occupation category (taking the first sentence on their page)

In [ ]:
with open(DATA + 'semantics/cleaned/occupations.json', 'r') as infile:
    classes = json.load(infile)
with open(DATA + 'semantics/occupation_sents.json', 'r') as infile: 
    occ_sents = json.load(infile)
with open(DATA + 'semantics/person_occupation_sents.json', 'r') as infile: 
    person_occ_sents = json.load(infile)

In [ ]:
random.seed(0)
for c in classes: 
    print(c.upper())
    occ_list = [i for i in classes[c]['high'] if i in occ_sents]
    if len(occ_list) > 10: 
        print(', '.join(random.sample(occ_list, 10)))
    else: 
        print(', '.join(occ_list))
    print()

For the examples below, we choose ones that are not confusing and lightly edit them for clarity, e.g. "A person is a person who" becomes "This person is someone who" before showing them to crowdworkers. 

In [ ]:
random.seed(10)
for c in classes: 
    print(c.upper())
    class_sents = []
    for i in classes[c]['high']: 
        if i in person_occ_sents: 
            for sent in person_occ_sents[i]: 
                if c != 'Math & statistics': # need more examples
                    if len(sent.split()) >= 10: 
                        class_sents.append(sent)
                        break
                else: 
                    class_sents.append(sent)
    for sent in class_sents: 
        print(sent)
    print()

In [ ]:
for c in classes: 
    print([i for i in classes[c]['high'] if i in occ_sents and i.endswith('s')])

In [ ]:
total = 0
for occ in occ_sents: 
    total += len(occ_sents[occ])
print(total)
print(len(occ_sents))

### Human evaluation results

In [ ]:
df = pd.read_csv('../data/human_eval_occupations.csv')
df = df[df['Finished'] == 'True']

In [ ]:
questions = [x for x in df.columns.to_list() if '_' in x and 'Q' in x]
print(questions)
occ_cats = list(set([x.split('_')[0] for x in questions]))

In [ ]:
model_map_Q1 = {'1':'bert-default', '2': 'bert-zscore', '3': 'bert-base-prob', '4': 'bert-base-prob-zscore'}
model_map_Q2 = {'1': 'glove', '6':'bert-default', '2': 'bert-zscore', 
                '3': 'bert-base-prob', '4': 'bert-base-prob-zscore'}

In [ ]:
def compute_average_rank(question_ID, model_map): 
    for model_num in model_map: 
        columns = []
        for cat in occ_cats: 
            columns.append(cat + '_' + question_ID + '_' + model_num)
        print()
        print([cat[:5] for cat in occ_cats])
        print(df[columns].values)
        print(model_map[model_num], round(np.mean(df[columns].values.astype(int)), 3))
        mu = np.mean(df[columns].values.astype(int))
        sigma = np.std(df[columns].values.astype(int))
        N = len(occ_cats) * len(df[columns].values)
        start, end = stats.norm.interval(0.95, loc=mu, scale=sigma/math.sqrt(N))
        print('95\% CI:', round((end - start) /2, 3))

In [ ]:
print("PERSON EXPERIMENT")
compute_average_rank('Q1', model_map_Q1)
print()
print("OCCUPATION EXPERIMENT")
compute_average_rank('Q2', model_map_Q2)

In [ ]:
# https://www.wikiwand.com/en/Kendall's_W
kendalls_w = []
m = 3
n = 4
question_ID = 'Q1'
for cat in occ_cats: 
    columns = []
    for model_num in model_map_Q1: 
        columns.append(cat + '_' + question_ID + '_' + model_num)
    sums = df[columns].astype(int).sum().to_list()
    Rbar = sum(sums) / n
    S = sum([(sums[x] - Rbar) ** 2 for x in range(n)])
    W = (12 * S) / (m ** 2 * (n ** 3 - n))
    print(cat + '_' + question_ID, round(W, 3))
    kendalls_w.append(W)
print()
    
m = 3
n = 5
question_ID = 'Q2'
for cat in occ_cats: 
    columns = []
    for model_num in model_map_Q2: 
        columns.append(cat + '_' + question_ID + '_' + model_num)
    sums = df[columns].astype(int).sum().to_list()
    Rbar = sum(sums) / n
    S = sum([(sums[x] - Rbar) ** 2 for x in range(n)])
    W = (12 * S) / (m ** 2 * (n ** 3 - n))
    print(cat + '_' + question_ID, round(W, 3))
    kendalls_w.append(W)
print()
print("AVG KENDALLS W:", round(np.mean(kendalls_w), 3))